# Creating a Mesh Model — documentation examples

Companion notebook to the **Creating a Mesh Model** documentation page
(`docs/source/mesh.rst`). One section per code snippet / figure; running the
whole notebook regenerates every figure on that page (light **and** dark
variants) into `docs/img/`.

Requires only the core `stellar-spice` install.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

from pathlib import Path
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Figures are written straight into the documentation image directory with the
# exact filenames the .rst pages reference.
DOCS_IMG = Path("..") / ".." / "docs" / "img"

def save_doc_fig(name, make_fig):
    """Render ``make_fig()`` twice — default (light) style and dark style — and
    save as <name>.png / <name>_dark.png in docs/img."""
    for style, suffix in [("default", ""), ("dark_background", "_dark")]:
        with plt.style.context(style):
            fig = make_fig()
            fig.savefig(DOCS_IMG / f"{name}{suffix}.png", dpi=120, bbox_inches="tight",
                        facecolor=fig.get_facecolor())
            plt.close(fig)
    print(f"saved {name}.png / {name}_dark.png")


## Basic Icosphere Creation

In [2]:
from spice.models import IcosphereModel
from spice.spectrum import Blackbody

# Initialize a Blackbody model (for spectrum calculation)
bb = Blackbody()

# Create an icosphere model
m = IcosphereModel.construct(
    1000,  # Minimal number of vertices
    1.,     # Radius in solar radii
    1.,     # Mass in solar masses
    bb.to_parameters(),  # Parameters for the spectrum model
    bb.parameter_names   # Names of the parameters used to match between overridable parameters and mesh parameters
)

/Users/mjablons/code/spice/src/spice/models/mesh_model.py:331: UserWarning: If override_log_g is True, either parameter_names must include one of [logg,loggs,log_g,log_gs,log g,log gs,surface gravity,surface gravities,surface_gravity,surface_gravities], or log_g_index must be passed for log g to be used in the spectrum emulator.
  warnings.warn(f"If override_log_g is True, either parameter_names must include one of [" + ",".join(


[spice] IcosphereModel constructed in 0.8 s


## Adding Rotation

In [3]:
from spice.models.mesh_transform import add_rotation, evaluate_rotation

# Add rotation to the model
rm = add_rotation(
    m, # Model instance
    rotation_velocity=10., # Rotation velocity in km/s
    rotation_axis=jnp.array([0, 0, 1]) # Rotation axis (relative to the model center)
)  # Rotate around z-axis

t = 1. # time in seconds

# To get the rotated mesh at a specific time:
rotated_mesh = evaluate_rotation(rm, t)  # t is the time in seconds

## Visualizing the Mesh

Figure: `rotated_mesh.png` / `rotated_mesh_dark.png`

In [4]:
from spice.plots import plot_3D

save_doc_fig("rotated_mesh",
             lambda: plot_3D(rotated_mesh, 'los_velocities')[0])

saved rotated_mesh.png / rotated_mesh_dark.png


## Adding Pulsation

Figure: `pulsated_mesh.png` / `pulsated_mesh_dark.png`

The pulsation period and the time passed to `evaluate_pulsations` are in days:
the conversion of pulsation velocities to km/s assumes solRad/day.

In [5]:
from spice.models.mesh_transform import add_pulsation, evaluate_pulsations

# Add pulsation to the model
mp = add_pulsation(
    m, # Model instance
    1, # m order
    1, # l degree
    100., # pulsation period
    jnp.array([[0.5, 0.]]) # Fourier series parameters
)

t = 40. # time in days, the same unit as the pulsation period

# To get the pulsated mesh at a specific time:
pulsated_mesh = evaluate_pulsations(mp, t)  # t is the time

[spice] Pulsations evaluated in 2.0 s


In [6]:
save_doc_fig("pulsated_mesh",
             lambda: plot_3D(pulsated_mesh, 'los_velocities')[0])

saved pulsated_mesh.png / pulsated_mesh_dark.png


## Tilted Pulsation Axis

Figure: `pulsation_with_tilt.png` / `pulsation_with_tilt_dark.png`

In [7]:
untilted_m = add_pulsation(m, 0, 1, 2., jnp.array([[1e-1, 0.]]))
tilted_m = add_pulsation(m, 0, 1, 2., jnp.array([[1e-1, 0.]]),
                         pulsation_axes=jnp.array([0., 1., 0.]),
                         pulsation_angles=jnp.array([45.]))

untilted_pulsated_m = evaluate_pulsations(untilted_m, 0.5)
tilted_pulsated_m = evaluate_pulsations(tilted_m, 0.5)

def tilt_comparison_fig():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), subplot_kw={'projection': '3d'})
    plot_3D(untilted_pulsated_m, property='los_velocities', axes=(fig, ax1))
    ax1.set_title('Untilted Pulsation')
    plot_3D(tilted_pulsated_m, property='los_velocities', axes=(fig, ax2))
    ax2.set_title('Tilted Pulsation')
    fig.tight_layout()
    return fig

save_doc_fig("pulsation_with_tilt", tilt_comparison_fig)

[spice] Pulsations evaluated in 0.0 s
[spice] Pulsations evaluated in 0.2 s
saved pulsation_with_tilt.png / pulsation_with_tilt_dark.png


## Adding Temperature Spots — spherical harmonic

Figure: `temp_harmonic.png` / `temp_harmonic_dark.png`

In [8]:
from spice.models.spots import add_spherical_harmonic_spot

base_temp = 5700
spot_temp = 15000

# Add temperature spots
m_harm_spot = add_spherical_harmonic_spot(
    m, # Model instance
    4, # m order
    4, # l degree
    param_delta=9300, # difference in the parameter value between the spot and the background
    param_index=0 # index of the parameter in the parameters array
)

save_doc_fig("temp_harmonic",
             lambda: plot_3D(m_harm_spot, property=0, property_label='Temperature [K]')[0])

saved temp_harmonic.png / temp_harmonic_dark.png


## Tilted spherical-harmonic spot

Figure: `tilted_temperature_spot.png` / `tilted_temperature_spot_dark.png`

In [9]:
m_harm_spot_tilted = add_spherical_harmonic_spot(
    m, # Model instance
    4, # m order
    4, # l degree
    param_delta=9300, # difference in the parameter value between the spot and the background
    param_index=0, # index of the parameter in the parameters array
    tilt_axis=jnp.array([0., 1., 0.]),
    tilt_angle=45.  # tilt angle in degrees
)

save_doc_fig("tilted_temperature_spot",
             lambda: plot_3D(m_harm_spot_tilted, property=0, property_label='Temperature [K]')[0])

saved tilted_temperature_spot.png / tilted_temperature_spot_dark.png


## Circular spot

Figure: `temp_spot.png` / `temp_spot_dark.png`

In [10]:
from spice.models.spots import add_spot

m_spot = add_spot(
    mesh=m, # Model instance
    spot_center_theta=0.5, # spot center in spherical coordinates, in radians
    spot_center_phi=0.5, # spot center in spherical coordinates, in radians
    spot_radius=50., # spot radius in degrees
    parameter_delta=1000, # difference in the parameter value between the spot and the background
    parameter_index=0, # index of the parameter in the parameters array
    smoothness=0.1 # smoothness of the spot edges
)

save_doc_fig("temp_spot",
             lambda: plot_3D(m_spot, property=0, property_label='Temperature [K]')[0])

saved temp_spot.png / temp_spot_dark.png


## Two spherical-harmonic modes

Figures: `temp_2_2_3_3.png` (+`_dark`) and the side-by-side
`temp_2_2_3_3_comparison.png` (+`_dark`)

In [11]:
from spice.models.spots import add_spherical_harmonic_spots

m_harm_spots = add_spherical_harmonic_spots(
    mesh=m,
    m_orders=jnp.array([2, 3.]),
    n_degrees=jnp.array([2, 3.]),
    param_deltas=jnp.array([1000, 1000.]),
    param_indices=jnp.array([0, 0])
)

save_doc_fig("temp_2_2_3_3",
             lambda: plot_3D(m_harm_spots, property=0, property_label='Temperature [K]')[0])

saved temp_2_2_3_3.png / temp_2_2_3_3_dark.png


In [12]:
m_22 = add_spherical_harmonic_spot(m, 2, 2, param_delta=1000, param_index=0)
m_33 = add_spherical_harmonic_spot(m, 3, 3, param_delta=1000, param_index=0)

def harmonic_comparison_fig():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), subplot_kw={'projection': '3d'})
    plot_3D(m_22, property=0, property_label='Temperature [K]', axes=(fig, ax1))
    ax1.set_title('m=2, l=2')
    plot_3D(m_33, property=0, property_label='Temperature [K]', axes=(fig, ax2))
    ax2.set_title('m=3, l=3')
    fig.tight_layout()
    return fig

save_doc_fig("temp_2_2_3_3_comparison", harmonic_comparison_fig)

saved temp_2_2_3_3_comparison.png / temp_2_2_3_3_comparison_dark.png


## Two circular spots

Figure: `temp_two_spots.png` / `temp_two_spots_dark.png`

In [13]:
from spice.models.spots import add_spots

m_spots = add_spots(
    mesh=m, # Model instance
    spot_center_thetas=jnp.array([0.5, 1.5]), # spot center in spherical coordinates, in radians
    spot_center_phis=jnp.array([0.5, 0.]), # spot center in spherical coordinates, in radians
    spot_radii=jnp.array([20., 20.]), # spot radius in degrees
    parameter_deltas=jnp.array([1000., 1000.]), # difference in the parameter value between the spot and the background
    parameter_indices=jnp.array([0, 0]), # index of the parameter in the parameters array
    smoothness=jnp.array([0.5, 0.5]) # smoothness of the spot edges
)

save_doc_fig("temp_two_spots",
             lambda: plot_3D(m_spots, property=0, property_label='Temperature [K]')[0])

saved temp_two_spots.png / temp_two_spots_dark.png
